# Exercise 2: how precise is the prototype? ★★☆☆☆

*Covers notebooks 2 and 3.*

The scatter of a constant star's light curve sets the smallest variability
you can detect. Measured for every star in a field, it traces out the
**noise curve**: the scatter against magnitude, flat for bright stars
(systematics) and rising for faint ones (photon and sky noise).

**Goal:** measure the ρ noise curve of segment 391749 from all its
minipixes, and save it for exercise 3.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices
GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# The staged sample. Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

from astropy.table import Table

FIELD = SAMPLE / "seg_391749"

In [ ]:
from astropy.stats import SigmaClip
from astropy.table import Table
from photutils.aperture import ApertureStats, CircularAnnulus, CircularAperture, aperture_photometry

EXPTIME = 60.0                                # s, every frame
SEGMENT_PIXEL, DETECTOR_PIXEL = 0.879, 1.012  # arcsec
R = 4 * DETECTOR_PIXEL / SEGMENT_PIXEL        # the zeropoints' 4.05" aperture, in segment pixels
ANNULUS = (R + 3, 15.5)                       # what fits in a 32-pixel cutout


def aperture_flux(cube, x, y, radius=R, annulus=ANNULUS):
    # Background-subtracted flux (e-/s) and its error in each plane of a cube,
    # with the background the sigma-clipped median of the annulus.
    aperture = CircularAperture((x, y), r=radius)
    ring = CircularAnnulus((x, y), r_in=annulus[0], r_out=annulus[1])
    clip = SigmaClip(sigma=3.0)
    flux, err = np.full(len(cube), np.nan), np.full(len(cube), np.nan)
    for k, plane in enumerate(np.asarray(cube, np.float64)):
        sky_ = ApertureStats(plane, ring, sigma_clip=clip)
        phot = aperture_photometry(plane - sky_.median, aperture,
                                   error=np.full(plane.shape, sky_.std), method="exact")
        flux[k] = phot["aperture_sum"][0]
        # the sky noise from the annulus, plus the source's photon noise
        err[k] = np.hypot(phot["aperture_sum_err"][0], np.sqrt(max(flux[k], 0) / EXPTIME))
    return flux, err


def light_curve(frames, epochs, flux, err, zp, flags=None):
    # A table of calibrated magnitudes; a frame is good when it is calibrated,
    # has a positive flux and (with DQ flags) none of argex.dq.BAD under the aperture.
    mag = zp - 2.5 * np.log10(flux)
    good = np.isfinite(mag)
    if flags is not None:
        good &= (flags & int(argex.dq.BAD)) == 0
    return Table({
        "epoch": epochs, "mjd": times.to_mjd(epochs),
        "frame_key": [f.frame_key for f in frames],
        "flux": flux, "flux_err": err, "zp": zp,
        "mag": mag, "mag_err": 2.5 / np.log(10) * err / flux,
        "dq": np.zeros(len(flux), np.uint32) if flags is None else flags, "good": good,
    })

def star_position(s):
    # brightest pixel of the median cutout, refined by a centroid
    med = np.nanmedian(s.data, axis=0)
    py, px = np.unravel_index(np.nanargmax(med), med.shape)
    yy, xx = np.mgrid[:32, :32]
    w = np.clip(med - np.nanmedian(med), 0, None) * ((xx - px) ** 2 + (yy - py) ** 2 <= 9)
    return (w * xx).sum() / w.sum(), (w * yy).sum() / w.sum(), np.nanmax(med)

## 1. One star

Write `noise(s)`: for a `MinipixSeries` of one band, find the star with
`star_position()`, measure it with `aperture_flux()`, calibrate it with the
zeropoints at the star (the pivot color is fine) and drop frames with
`argex.dq.BAD` under the aperture. Return the median magnitude and a robust
scatter (1.4826 × the median absolute deviation, in mag). Skip stars too
close to the cutout's edge, and series with few frames.

In [ ]:
def noise(s):
    ...
    return median_mag, scatter, n_good

## 2. Every star

Run it on the ρ frames of every minipix in the segment (about a minute),
plot the scatter against magnitude, and save the table as
`noise_rho.ecsv` for exercise 3.

In [ ]:
rows = []
for path in sorted((FIELD / "minipix").glob("minipix_*.fits")):
    ...

## 3. Questions

* What scatter does the prototype reach at its brightest, and where does the
  curve start to rise?
* Which stars sit far above the curve? Look at their light curves: variables,
  blends, or something else?